# Normalizing Flows

A self-contained refresher: how normalizing flows turn a simple base distribution
into a complex one through an **invertible, differentiable** map, and why that buys you
something diffusion, VAEs, and GANs don't — an *exact* tractable likelihood.

**Domain:** Architectures  ·  **recommended addition**  ·  **runnable:** yes

## 1. What & Why

**What it is.** A normalizing flow (NF) is a generative model built from a chain of
**bijections** (invertible, differentiable maps) that transform a simple base
distribution $p_Z$ — almost always a standard Gaussian — into the data distribution
$p_X$. Because each step is invertible and we can compute its Jacobian determinant, the
**change-of-variables formula** gives the exact density of any data point:

$$\log p_X(x) = \log p_Z\big(f(x)\big) + \log\left|\det \frac{\partial f}{\partial x}\right|$$

where $f$ maps data $\to$ base (the *normalizing* direction) and $f^{-1}$ maps base $\to$
data (the *generating* direction).

**The problem it solves.** Most deep generative models give up exact likelihood to gain
flexibility:

- **VAEs** optimize a *lower bound* (the ELBO), not the true likelihood.
- **GANs** have no likelihood at all — you can sample but not score.
- **Autoregressive models** give exact likelihood but sample slowly, one dimension at a time.

Flows are the model that gives you **both** exact log-likelihood *and* one-shot sampling
from the same network — the price is an architecture constrained to be invertible with a
cheap Jacobian determinant.

**When to reach for it.** Density estimation; exact likelihood for out-of-distribution /
anomaly detection; flexible variational posteriors (normalizing-flow posteriors in VI);
invertible representation learning; simulation-based / scientific inference. **When not:**
high-resolution image synthesis (flows are dimension-preserving and memory-hungry, and
diffusion/GANs produce sharper samples) — reach for [`diffusion-models`](diffusion-models.ipynb)
or [`flow-matching`](flow-matching.ipynb) there.

## 2. Mental Model

Think of a stack of **invertible lenses**. You start with a featureless blob of
Gaussian "probability clay". Each lens warps space — stretching it here, compressing it
there — and after the whole stack the clay has been molded into the shape of your data.

> **Probability mass is conserved; only its density changes as volume stretches or shrinks.**

That conservation is the whole game. When a map compresses a region (shrinks volume), the
density there must go *up* to keep total mass at 1; when it stretches a region, density
goes *down*. The **Jacobian determinant** is exactly the local volume-change factor, so
the log-density correction is $\log|\det J|$. Evaluating a data point's likelihood is
just: *run it backwards through the lenses to the Gaussian, read off the Gaussian density,
and add up how much each lens stretched space along the way.*

## 3. Key Concepts

- **Bijector / diffeomorphism.** Each layer is invertible and differentiable. Composing
  bijectors gives a bijector, so deep flows stay invertible.
- **Base distribution $p_Z$.** The simple, known distribution (standard Gaussian) you can
  sample and evaluate trivially.
- **Change of variables.** $\log p_X(x) = \log p_Z(f(x)) + \log|\det \partial f/\partial x|$.
  The entire model is trained by maximizing this exact log-likelihood.
- **Log-det Jacobian.** The volume-change term. A flow is only practical if this is cheap
  to compute — the central design constraint.
- **Coupling layers (RealNVP/NICE).** Split the input in two halves; pass one half through
  unchanged and use it to predict an affine (scale+shift) transform of the other half. The
  Jacobian is triangular, so its log-det is just the sum of the scale terms — $O(d)$.
- **Autoregressive flows (MAF / IAF).** Each dimension transformed conditioned on previous
  ones; also triangular Jacobian. **MAF** has fast density / slow sampling; **IAF** is the
  reverse — same math, opposite speed trade-off.
- **Permutations / masking.** Coupling leaves half the dims untouched per layer, so you
  must alternate which half is fixed (or insert learned permutations, as in Glow) or some
  dimensions never get transformed.
- **Dimension preservation.** Latent and data dimensions are equal — flows never compress,
  which is what makes invertibility (and exact likelihood) possible.

## 4. Setup

Everything below is tiny and **CPU-friendly**: an analytic 1-D demo with only `numpy`,
then a small RealNVP coupling flow (a handful of MLPs) trained on 2-D toy data with
`torch`. Install into a fresh environment if needed (uncomment), otherwise the imports
are all you need.

In [1]:
# %pip install torch numpy matplotlib
import os
import math
import numpy as np
import torch
import torch.nn as nn

rng = np.random.default_rng(0)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)

torch 2.12.1 | device: cpu


## 5. Worked Examples

### Example 1 — Change of variables, by hand (1-D affine flow)

The simplest possible flow: a single affine bijector $x = \sigma z + \mu$ with base
$z \sim \mathcal{N}(0,1)$. The *normalizing* map is $f(x) = (x-\mu)/\sigma$ and
$\log|\mathrm{d}f/\mathrm{d}x| = -\log\sigma$. Plugging into change of variables should
reproduce a $\mathcal{N}(\mu,\sigma)$ density exactly — and integrate to 1. This makes
the log-det term concrete before we stack many of them.

In [2]:
mu, sigma = 2.0, 0.5

def base_logpdf(z):                 # standard normal log-density
    return -0.5 * (z**2 + np.log(2 * np.pi))

def normalize(x):                   # data -> base   (the f direction)
    return (x - mu) / sigma

log_det = -np.log(sigma)            # log|df/dx|, constant for an affine map

def model_logpdf(x):                # change of variables
    return base_logpdf(normalize(x)) + log_det

# Sanity checks: integrates to 1, and matches the analytic Normal(mu, sigma).
grid = np.linspace(-2.0, 6.0, 8001)
px = np.exp(model_logpdf(grid))
dx = grid[1] - grid[0]
integral = np.trapezoid(px, dx=dx) if hasattr(np, 'trapezoid') else np.trapz(px, dx=dx)
analytic = np.exp(-0.5 * ((grid - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))
print(f"integral of p(x) dx = {integral:.6f}  (should be 1)")
print(f"max |model - analytic| = {np.abs(px - analytic).max():.2e}")

integral of p(x) dx = 1.000000  (should be 1)
max |model - analytic| = 2.22e-16


The density integrates to 1 and matches the closed-form Gaussian to machine precision —
the $-\log\sigma$ term is doing exactly its job. Compressing space ($\sigma<1$) raises the
density. Now we let a neural net learn the warps instead of hand-picking them.

### Example 2 — Train a RealNVP coupling flow on 2-D data

Target: the classic **two-moons**. We stack affine coupling layers (RealNVP). Each layer
passes one coordinate through unchanged and uses it to predict a `scale, shift` for the
other; we alternate which coordinate is fixed. The Jacobian is triangular, so its log-det
is just the sum of the predicted log-scales. We train by **maximizing exact log-likelihood**
(minimizing NLL).

In [3]:
def two_moons(n, noise=0.08):
    """Classic two-moons; standardized to ~unit scale for stable training."""
    t = rng.uniform(0, np.pi, size=n)
    half = n // 2
    x = np.empty((n, 2))
    x[:half] = np.stack([np.cos(t[:half]), np.sin(t[:half])], axis=1)
    x[half:] = np.stack([1 - np.cos(t[half:]), 0.5 - np.sin(t[half:])], axis=1)
    x += noise * rng.standard_normal(x.shape)
    x = (x - x.mean(0)) / x.std(0)
    return torch.tensor(x, dtype=torch.float32)


class AffineCoupling(nn.Module):
    """RealNVP affine coupling. `mask` is 1 on pass-through dims, 0 on transformed dims."""
    def __init__(self, mask, hidden=64):
        super().__init__()
        self.register_buffer('mask', mask)
        self.net = nn.Sequential(
            nn.Linear(2, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 4),          # -> (log_scale, shift) for both dims
        )

    def _scale_shift(self, x_masked):
        s, t = self.net(x_masked).chunk(2, dim=1)
        s = torch.tanh(s)                  # bound the scale for numerical stability
        return s * (1 - self.mask), t * (1 - self.mask)

    def forward(self, x):                  # data -> latent, returns log|det|
        xm = x * self.mask
        s, t = self._scale_shift(xm)
        z = xm + (1 - self.mask) * (x * torch.exp(s) + t)
        return z, s.sum(dim=1)

    def inverse(self, z):                  # latent -> data
        zm = z * self.mask
        s, t = self._scale_shift(zm)       # pass-through dims are identical, so use z
        return zm + (1 - self.mask) * ((z - t) * torch.exp(-s))


class RealNVP(nn.Module):
    def __init__(self, n_layers=8, hidden=64):
        super().__init__()
        masks = [torch.tensor([1.0, 0.0]) if i % 2 == 0 else torch.tensor([0.0, 1.0])
                 for i in range(n_layers)]
        self.layers = nn.ModuleList([AffineCoupling(m, hidden) for m in masks])

    def log_prob(self, x):
        z, log_det = x, torch.zeros(x.shape[0])
        for layer in self.layers:
            z, ld = layer(z)
            log_det = log_det + ld
        base = -0.5 * (z**2 + math.log(2 * math.pi)).sum(dim=1)
        return base + log_det

    @torch.no_grad()
    def sample(self, n):
        x = torch.randn(n, 2)
        for layer in reversed(self.layers):
            x = layer.inverse(x)
        return x


model = RealNVP().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for step in range(2001):
    x = two_moons(512)
    loss = -model.log_prob(x).mean()       # negative log-likelihood
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 500 == 0:
        print(f'step {step:4d}  NLL {loss.item():.3f}')

step    0  NLL 3.289


step  500  NLL 1.847


step 1000  NLL 1.761


step 1500  NLL 1.731


step 2000  NLL 1.723


NLL drops steadily — the flow is learning to assign high likelihood to the moons. Because
the model is a true density, we can do two things a GAN can't: **score** any point's
log-likelihood, and **sample** by pushing Gaussian noise through the inverse map. Let's
check both, then visualize.

In [4]:
# Score: real moon points should have much higher log-likelihood than off-manifold noise.
real = two_moons(2000)
junk = torch.randn(2000, 2) * 3.0
print(f'mean log p(real) = {model.log_prob(real).mean().item():.2f}')
print(f'mean log p(junk) = {model.log_prob(junk).mean().item():.2f}')

# Sample by inverting the flow from Gaussian noise.
gen = model.sample(2000)
print('generated mean/std:', gen.mean(0).tolist(), gen.std(0).tolist())
print('real      mean/std:', real.mean(0).tolist(), real.std(0).tolist())

mean log p(real) = -1.76
mean log p(junk) = -94.62
generated mean/std: [-0.07705847918987274, 0.034579575061798096] [1.0044772624969482, 1.0120123624801636]
real      mean/std: [3.814697446813398e-09, -7.629394893626795e-09] [1.0002501010894775, 1.0002501010894775]


In [5]:
import matplotlib
matplotlib.use('Agg')   # headless: keeps the notebook executable anywhere
import matplotlib.pyplot as plt

real_np, gen_np = real.numpy(), gen.detach().numpy()
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].scatter(real_np[:, 0], real_np[:, 1], s=3, alpha=0.4, color='tab:blue')
ax[0].set_title('Target  (two moons)')
ax[1].scatter(gen_np[:, 0], gen_np[:, 1], s=3, alpha=0.4, color='tab:green')
ax[1].set_title('Generated  (Gaussian -> inverse flow)')
for a in ax:
    a.set_xlim(-2.5, 2.5); a.set_ylim(-2.5, 2.5); a.set_aspect('equal')
plt.tight_layout()
plt.savefig('normalizing_flows_samples.png', dpi=80)
print('saved normalizing_flows_samples.png  | generated moon shape matches target')

saved normalizing_flows_samples.png  | generated moon shape matches target


### (Optional) Use a maintained library

For real work, don't hand-roll coupling layers — use a tested library with permutations,
batch-norm flows, spline couplings, and multi-scale architectures. The cell below is
**gated** behind an env var so the notebook still runs without the extra dependency.

In [6]:
if os.getenv('RUN_NF_LIB'):
    # pip install normflows
    import normflows as nf
    base = nf.distributions.DiagGaussian(2)
    flows = []
    for _ in range(8):
        param_map = nf.nets.MLP([1, 64, 64, 2], init_zeros=True)
        flows.append(nf.flows.AffineCouplingBlock(param_map))
        flows.append(nf.flows.Permute(2, mode='swap'))
    nfm = nf.NormalizingFlow(base, flows)
    x = two_moons(256)
    print('library NLL:', nfm.forward_kld(x).item())
else:
    print("set RUN_NF_LIB=1 to run the `normflows` example")

set RUN_NF_LIB=1 to run the `normflows` example


## 6. Gotchas & Pitfalls

- **Dimension preserving = expensive.** Latent dim equals data dim, always. For images
  that means the flow carries the full pixel count through every layer — memory-heavy and
  why flows lost the image race to diffusion.
- **Alternate the coupling masks.** A coupling layer leaves half the dimensions untouched.
  If you don't flip which half is fixed (or insert permutations), some dims never get
  transformed and the model can't fit. We alternate `[1,0]`/`[0,1]` above.
- **Bound the scale.** Unbounded `exp(s)` blows up the log-det and the loss. Squash the
  predicted log-scale (we use `tanh`) or you'll get NaNs early in training.
- **The log-det must stay cheap.** A general $d\times d$ Jacobian determinant is $O(d^3)$.
  Coupling and autoregressive flows are popular precisely because their Jacobian is
  triangular, making the log-det an $O(d)$ sum. Don't design a layer whose Jacobian you
  can't cheaply compute.
- **Forward/inverse speed asymmetry.** Coupling flows are fast both ways; autoregressive
  flows are not. **MAF**: fast likelihood, slow sampling. **IAF**: slow likelihood, fast
  sampling. Pick the orientation that matches your hot path.
- **Dequantize discrete data.** Likelihood on a continuous model fit to integer pixel
  values can be driven to $+\infty$ by collapsing onto the grid. Add uniform noise
  (dequantization) before training on images.
- **Per-layer expressivity is limited.** Invertibility with a cheap Jacobian constrains
  each layer, so flows need *many* layers to be expressive — depth is not optional.

## 7. When to Use vs Alternatives

| Model | Likelihood | Sampling | Best when |
|---|---|---|---|
| **Normalizing Flow** | **Exact**, tractable | One-shot (invert the map) | You need a real density: OOD/anomaly detection, scoring, flexible VI posteriors, scientific inference |
| **VAE** ([`vae`](vae.ipynb)) | Lower bound (ELBO) | One-shot from decoder | You want a compact latent code and amortized inference; exact likelihood not required |
| **GAN** ([`gan`](gan.ipynb)) | None | One-shot, sharp | Sample quality is everything and you don't need to score points |
| **Diffusion** ([`diffusion-models`](diffusion-models.ipynb)) | Bound / via ODE | Many steps (iterative) | State-of-the-art image/audio synthesis; mature conditioning ecosystem |
| **Flow Matching** ([`flow-matching`](flow-matching.ipynb)) | Via the ODE (CNF) | Few-step ODE | Continuous data, fast deterministic sampling, simple regression objective |

**Rule of thumb:** if you genuinely need an *exact, evaluable density* (not just samples),
normalizing flows are the natural choice. If you mostly need great samples, a diffusion or
flow-matching model will usually beat a flow of comparable size — and "continuous
normalizing flows" trained by flow matching recover much of the flow elegance without the
per-layer Jacobian constraint.

## 8. Resources

- **Dinh et al., "Density estimation using Real NVP" (2017)** — the affine coupling flow
  implemented above: https://arxiv.org/abs/1605.08803
- **Dinh et al., "NICE" (2014)** — the original additive coupling flow:
  https://arxiv.org/abs/1410.8516
- **Papamakarios et al., "Masked Autoregressive Flow" (2017)** — MAF/IAF autoregressive
  flows: https://arxiv.org/abs/1705.07057
- **Kingma & Dhariwal, "Glow" (2018)** — invertible 1x1 convolutions, multi-scale flows:
  https://arxiv.org/abs/1807.03039
- **Papamakarios et al., "Normalizing Flows for Probabilistic Modeling and Inference"
  (2021)** — the definitive survey: https://arxiv.org/abs/1912.02762
- **Lilian Weng, "Flow-based Deep Generative Models"** — clear visual tutorial:
  https://lilianweng.github.io/posts/2018-10-13-flow-models/
- **`normflows` library (GitHub)** — tested flows, couplings, and base distributions:
  https://github.com/VincentStimper/normalizing-flows